In [0]:
# Define variables
storage_account_name = "datastorejimi"
container_name = "bronze"
mount_point = f"/mnt/{container_name}"

# Set up the configuration
dbutils.fs.mount(
  source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
  mount_point = mount_point,
  extra_configs = {
  }
)


True

In [0]:
# List files in the mounted path
dbutils.fs.ls("/mnt/bronze")

[FileInfo(path='dbfs:/mnt/bronze/Categories.csv', name='Categories.csv', size=2615, modificationTime=1743029156000),
 FileInfo(path='dbfs:/mnt/bronze/Geolocation.csv', name='Geolocation.csv', size=61273883, modificationTime=1743029164000),
 FileInfo(path='dbfs:/mnt/bronze/Order Items.csv', name='Order Items.csv', size=15438671, modificationTime=1743029163000),
 FileInfo(path='dbfs:/mnt/bronze/Order Payments.csv', name='Order Payments.csv', size=5777138, modificationTime=1743029154000),
 FileInfo(path='dbfs:/mnt/bronze/Orders.csv', name='Orders.csv', size=15315857, modificationTime=1743029159000),
 FileInfo(path='dbfs:/mnt/bronze/Products.csv', name='Products.csv', size=2379446, modificationTime=1743029153000),
 FileInfo(path='dbfs:/mnt/bronze/Reviews.csv', name='Reviews.csv', size=14451670, modificationTime=1743029156000),
 FileInfo(path='dbfs:/mnt/bronze/Sellers.csv', name='Sellers.csv', size=174703, modificationTime=1743029152000),
 FileInfo(path='dbfs:/mnt/bronze/customers.csv', nam

In [0]:
!ls -lha /mnt/bronze

ls: cannot access '/mnt/bronze': No such file or directory


In [0]:
# Read CSV into DataFrame
df_customers = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/mnt/bronze/customers.csv")

# Show the DataFrame
df_customers.show(5)
df_customers.printSchema()


+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows
root
 |-- customer_id: string (

In [0]:
#Create a directory for your Bronze Delta Table
bronze_path = "/mnt/bronze/bronze_customers"

# Save as Delta table
df_customers.write.format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

In [0]:
#Register the Bronze Delta Table in the Metastore 

# Create a database for organizing your tables
spark.sql("CREATE DATABASE IF NOT EXISTS retail")

# Register the delta table under the database
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS retail.customers_bronze
    USING DELTA
    LOCATION '{bronze_path}'
""")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-3552796256948620>, line 7
      4 spark.sql("CREATE DATABASE IF NOT EXISTS retail")
      6 # Register the delta table under the database
----> 7 spark.sql(f"""
      8     CREATE TABLE IF NOT EXISTS retail.customers_bronze
      9     USING DELTA
     10     LOCATION '{bronze_path}'
     11 """)

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databricks/spark/python/pyspark/sql/session.py:1854, in SparkSession.sql(self, sqlQuery, args, **kwargs)
   1849     else:
   1850         raise PySparkType

In [0]:
# Save the DataFrame as a managed Delta table directly
df_customers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail.customers_bronze")

In [0]:
%sql

SELECT * FROM retail.customers_bronze LIMIT 10

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [0]:
#Let's read from the Bronze table and clean it
# Read from the Bronze Delta Table
df_bronze = spark.read.table("retail.customers_bronze")

# Clean and transform
from pyspark.sql.functions import col, upper, trim



In [0]:
df_silver = df_bronze.dropDuplicates(["customer_unique_id"]) \
    .filter(col("customer_zip_code_prefix").isNotNull()) \
    .withColumn("customer_city", upper(trim(col("customer_city")))) \
    .withColumn("customer_state", upper(trim(col("customer_state"))))



+--------------------+--------------------+------------------------+--------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix| customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------+--------------+
|3b37fb626fdf46cd9...|0005e1862207bf6cc...|                   25966|   TERESOPOLIS|            RJ|
|2f29573c8cac5a7be...|0006fdc98a402fceb...|                   29400| MIMOSO DO SUL|            ES|
|40f0183f7439212e8...|00090324bbad0e934...|                   13054|      CAMPINAS|            SP|
|0e114b02a45c98760...|000c8bdb58a29e711...|                   31555|BELO HORIZONTE|            MG|
|064064dd94c430137...|00115fc7123b5310c...|                   71015|      BRASILIA|            DF|
+--------------------+--------------------+------------------------+--------------+--------------+
only showing top 5 rows


In [0]:
df_silver.show(5)

+--------------------+--------------------+------------------------+--------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix| customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------+--------------+
|3b37fb626fdf46cd9...|0005e1862207bf6cc...|                   25966|   TERESOPOLIS|            RJ|
|2f29573c8cac5a7be...|0006fdc98a402fceb...|                   29400| MIMOSO DO SUL|            ES|
|40f0183f7439212e8...|00090324bbad0e934...|                   13054|      CAMPINAS|            SP|
|0e114b02a45c98760...|000c8bdb58a29e711...|                   31555|BELO HORIZONTE|            MG|
|064064dd94c430137...|00115fc7123b5310c...|                   71015|      BRASILIA|            DF|
+--------------------+--------------------+------------------------+--------------+--------------+
only showing top 5 rows


In [0]:
#Let’s save the cleaned dataset as your Silver Delta Table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail.customers_silver")

In [0]:
#Query the Silver Table Run a SQL query to confirm

In [0]:
%sql
SELECT customer_state, COUNT(*) AS total_customers
FROM retail.customers_silver
GROUP BY customer_state
ORDER BY total_customers DESC

customer_state,total_customers
SP,40293
RJ,12378
MG,11254
RS,5277
PR,4881
SC,3529
BA,3277
DF,2073
ES,1964
GO,1949
